# Weight Initialization and Training Stability Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Initialization Strategies

Four ways to initialize a weight matrix. Each returns a list of lists (a 2D matrix) with fan_in columns and fan_out rows.

In [ ]:
```python

import math

import random

def zero_init(fan_in, fan_out):

    return [[0.0 for _ in range(fan_in)] for _ in range(fan_out)]

def random_init(fan_in, fan_out, scale=1.0):

    return [[random.gauss(0, scale) for _ in range(fan_in)] for _ in range(fan_out)]

def xavier_init(fan_in, fan_out):

    std = math.sqrt(2.0 / (fan_in + fan_out))

    return [[random.gauss(0, std) for _ in range(fan_in)] for _ in range(fan_out)]

def kaiming_init(fan_in, fan_out):

    std = math.sqrt(2.0 / fan_in)

    return [[random.gauss(0, std) for _ in range(fan_in)] for _ in range(fan_out)]

In [ ]:
```

### Step 2: Activation Functions

We need sigmoid, tanh, and ReLU to test each init strategy with its intended activation.

In [ ]:
```python

def sigmoid(x):

    x = max(-500, min(500, x))

    return 1.0 / (1.0 + math.exp(-x))

def tanh_act(x):

    return math.tanh(x)

def relu(x):

    return max(0.0, x)

In [ ]:
```

### Step 3: Forward Pass Through 50 Layers

Pass random data through a deep network and measure mean activation magnitude at each layer.

In [ ]:
```python

def forward_deep(init_fn, activation_fn, n_layers=50, width=64, n_samples=100):

    random.seed(42)

    layer_magnitudes = []

    inputs = [[random.gauss(0, 1) for _ in range(width)] for _ in range(n_samples)]

    for layer_idx in range(n_layers):

        weights = init_fn(width, width)

        biases = [0.0] * width

        new_inputs = []

        for sample in inputs:

            output = []

            for neuron_idx in range(width):

                z = sum(weights[neuron_idx][j] * sample[j] for j in range(width)) + biases[neuron_idx]

                output.append(activation_fn(z))

            new_inputs.append(output)

        inputs = new_inputs

        magnitudes = []

        for sample in inputs:

            magnitudes.append(sum(abs(v) for v in sample) / width)

        mean_mag = sum(magnitudes) / len(magnitudes)

        layer_magnitudes.append(mean_mag)

    return layer_magnitudes

In [ ]:
```

### Step 4: The Experiment

Run all combinations: zero init, random N(0,1), random N(0,0.01), Xavier with sigmoid, Xavier with tanh, Kaiming with ReLU. Print the magnitude at key layers.

In [ ]:
```python

def run_experiment():

    configs = [

        ("Zero init + Sigmoid", lambda fi, fo: zero_init(fi, fo), sigmoid),

        ("Random N(0,1) + ReLU", lambda fi, fo: random_init(fi, fo, 1.0), relu),

        ("Random N(0,0.01) + ReLU", lambda fi, fo: random_init(fi, fo, 0.01), relu),

        ("Xavier + Sigmoid", xavier_init, sigmoid),

        ("Xavier + Tanh", xavier_init, tanh_act),

        ("Kaiming + ReLU", kaiming_init, relu),

    ]

    print(f"{'Strategy':<30} {'L1':>10} {'L5':>10} {'L10':>10} {'L25':>10} {'L50':>10}")

    print("-" * 80)

    for name, init_fn, act_fn in configs:

        mags = forward_deep(init_fn, act_fn)

        row = f"{name:<30}"

        for idx in [0, 4, 9, 24, 49]:

            val = mags[idx]

            if val > 1e6:

                row += f" {'EXPLODED':>10}"

            elif val < 1e-6:

                row += f" {'VANISHED':>10}"

            else:

                row += f" {val:>10.4f}"

        print(row)

In [ ]:
```

### Step 5: Symmetry Demonstration

Show that zero init produces identical neurons.

In [ ]:
```python

def symmetry_demo():

    random.seed(42)

    weights = zero_init(2, 4)

    biases = [0.0] * 4

    inputs = [0.5, -0.3]

    outputs = []

    for neuron_idx in range(4):

        z = sum(weights[neuron_idx][j] * inputs[j] for j in range(2)) + biases[neuron_idx]

        outputs.append(sigmoid(z))

    print("\nSymmetry Demo (4 neurons, zero init):")

    for i, out in enumerate(outputs):

        print(f"  Neuron {i}: output = {out:.6f}")

    all_same = all(abs(outputs[i] - outputs[0]) < 1e-10 for i in range(len(outputs)))

    print(f"  All identical: {all_same}")

    print(f"  Effective parameters: 1 (not {len(weights) * len(weights[0])})")

In [ ]:
```

### Step 6: Layer-by-Layer Magnitude Report

Print a visual bar chart of activation magnitudes through 50 layers.

In [ ]:
```python

def magnitude_report(name, magnitudes):

    print(f"\n{name}:")

    for i, mag in enumerate(magnitudes):

        if i % 5 == 0 or i == len(magnitudes) - 1:

            if mag > 1e6:

                bar = "X" * 50 + " EXPLODED"

            elif mag < 1e-6:

                bar = "." + " VANISHED"

            else:

                bar_len = min(50, max(1, int(mag * 10)))

                bar = "#" * bar_len

            print(f"  Layer {i+1:3d}: {bar} ({mag:.6f})")

In [ ]:
```

## Exercises

In [ ]:
1. Add LeCun initialization (Var = 1/fan_in, designed for SELU activation). Run the 50-layer experiment with LeCun init + tanh and compare to Xavier + tanh.

2. Implement the GPT-2 residual scaling: multiply the output of each layer by 1/sqrt(2*N) before adding to the residual stream. Run 50 layers with and without scaling, measure how fast the residual magnitude grows.

3. Create an "init health check" function that takes a network's layer dimensions and activation type, then recommends the correct initialization and warns if the current init will cause problems.

4. Run the experiment with fan_in = 16 vs fan_in = 1024. Xavier and Kaiming adapt to fan_in, but random init doesn't. Show how the gap between "works" and "breaks" widens with larger layers.

5. Implement orthogonal initialization (generate a random matrix, compute its SVD, use the orthogonal matrix U). Compare to Kaiming for ReLU networks at 50 layers.